# Network analysis with innovation and sustainability data

**DAISY International Summer School 2026 — hands-on session (90 minutes)**
Fabrizio Fusillo, University of Turin

Patent data (OECD REGPAT), EU-funded project data (CORDIS Horizon Europe) and
publication data (OpenAlex) share one feature: the relation between actors is
never observed directly, it is *inferred* from co-participation in a document.
This session goes from those raw tables to networks, to positional measures,
and finally to the network-based indicators that end up in econometric models.

---
### Before you start (Colab only)
1. `Runtime` → `Change runtime type` → **R**
2. Run the setup cell below once (≈1 minute: it installs packages and
   downloads ~5 MB of data).
3. In VS Code / RStudio you do not need this notebook: open the `.R` scripts
   in `lesson/` and run them line by line.


## Setup — packages, data access and the three helper functions

Everything below is identical to `00_setup.R` in the repository.


In [ ]:
t0 <- Sys.time()

if (Sys.info()[["sysname"]] == "Linux") {
  codename <- tryCatch({
    os <- readLines("/etc/os-release", warn = FALSE)
    sub('.*=', '', grep("^VERSION_CODENAME=", os, value = TRUE))
  }, error = function(e) "jammy")
  if (length(codename) == 0 || codename == "") codename <- "jammy"
  options(repos = c(CRAN = sprintf(
    "https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)))
  ## (ii) the User-Agent that unlocks the binaries
  options(HTTPUserAgent = sprintf(
    "R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
  message("repo: ", getOption("repos")[["CRAN"]])
} else {
  options(repos = c(CRAN = "https://cloud.r-project.org"))
}
options(timeout = 1800)   # the 60s default is not enough to download bulk data
options(Ncpus = max(2L, parallel::detectCores(logical = TRUE)))

## Core first (13-24 packages), plotting second (27 more, almost all from
## ggraph's tidyverse dependencies). Installing in two steps does not make it
## faster, but it tells you where the time goes - and lets you start reading the
## data while the plotting stack lands.
install_phase <- function(pkgs, label) {
  new_pkgs <- setdiff(pkgs, rownames(installed.packages()))
  if (!length(new_pkgs)) { message("[", label, "] already installed"); return(invisible()) }
  message("[", label, "] installing: ", paste(new_pkgs, collapse = ", "))
  t <- Sys.time()
  install.packages(new_pkgs, quiet = TRUE)
  message("[", label, "] done in ", round(difftime(Sys.time(), t, units = "secs")), "s")
}
install_phase(c("data.table", "igraph", "Matrix", "jsonlite", "R.utils"), "core")
install_phase(c("ggplot2", "ggraph"), "plotting")

pkgs <- c("data.table",   # fast data handling (the workhorse for raw big files)
          "igraph",       # network analysis
          "Matrix",       # sparse matrices: two-mode -> one-mode projections
          "ggplot2",      # plots
          "ggraph",       # network visualisation, ggplot2 grammar
          "jsonlite",     # REST APIs (OpenAlex)
          "R.utils")      # fread() needs it to read .csv.gz on Colab: keep it
invisible(lapply(pkgs, library, character.only = TRUE))
message("setup: ", round(difftime(Sys.time(), t0, units = "secs")), "s in total")

## optional packages, only used in clearly marked "if you have time" chunks
## install.packages(c("sna", "intergraph", "graphlayouts"))

## ---------------------------------------------------------------------------
## 1b. If the install is slow: what to check (Colab)
## ---------------------------------------------------------------------------
## Run this to see whether you are getting binaries or building from source.
## "x-package-type: binary" = good; anything else means every compiled package
## is being built locally, which is what turns one minute into ten.
##
##   cat(R.version.string, "\n")
##   u <- paste0(getOption("repos")[["CRAN"]], "/src/contrib/igraph_2.3.3.tar.gz")
##   h <- curlGetHeaders(u, verify = FALSE)
##   grep("x-package-type|x-package-binary-tag", h, value = TRUE, ignore.case = TRUE)
##
## Even with binaries, 51 packages take a few minutes on a Colab CPU: the loop is
## dominated by one HTTP request plus one unpack per package, not by computation,
## which is why a "more powerful" runtime changes nothing. Two ways out:
##   - run this cell FIRST and let it work through the framing slides;
##   - or install into a mounted Drive folder once and reuse it across sessions:
##       dir.create("/content/drive/MyDrive/Rlib", showWarnings = FALSE)
##       .libPaths("/content/drive/MyDrive/Rlib")     # before install.packages()

setDTthreads(0)           # use all available cores
set.seed(20260907)        # layouts and community detection are stochastic

## ---------------------------------------------------------------------------
## 2. Where is the data?
## ---------------------------------------------------------------------------
## The session works with small pre-processed extracts (~5 MB in total) of
## four sources. Locally they sit in ./data ; in Colab they are downloaded once
## from the course repository. Everything is read through daisy_data().
DAISY_URL <- Sys.getenv("DAISY_DATA_URL",
  "https://raw.githubusercontent.com/ffusillo/daisy-networks/main/data/")

daisy_data <- function(file) {
  local <- file.path("data", file)
  if (file.exists(local)) return(local)
  local <- file.path("lesson", "data", file)
  if (file.exists(local)) return(local)
  cache <- file.path(tempdir(), "daisy_data")
  dir.create(cache, showWarnings = FALSE, recursive = TRUE)
  dest <- file.path(cache, file)
  if (!file.exists(dest)) {
    message("downloading ", file, " ...")
    download.file(paste0(DAISY_URL, file), dest, mode = "wb", quiet = TRUE)
  }
  dest
}

## ---------------------------------------------------------------------------
## 3. HELPER 1 - from affiliation (two-mode) data to a one-mode network
## ---------------------------------------------------------------------------
## Almost all innovation network data are *indirectly observed*: we do not see
## the tie, we see two actors sharing an event (a patent, a project, a paper).
## The event x actor incidence matrix B gives the one-mode projection
##      A = t(B) %*% B
## where A[i,j] = number of events shared by actors i and j, and A[i,i] = number
## of events of actor i. Sparse matrices make this cheap even for 10^5 actors.
##
##   dt     : data.table in long format, one row = one actor in one event
##   event  : name of the event column  (patent, project, publication ...)
##   actor  : name of the actor column  (inventor, organisation, institution...)
##   max_size: drop events with more actors than this (huge events create huge
##             cliques: 1 project with 200 partners = 19,900 edges)
proj_two_mode <- function(dt, event, actor, max_size = Inf) {
  d <- unique(as.data.table(dt)[, .(ev = get(event), ac = get(actor))])
  d <- d[!is.na(ev) & !is.na(ac) & ev != "" & ac != ""]
  if (is.finite(max_size)) {
    big <- d[, .N, by = ev][N > max_size, ev]
    if (length(big)) message("dropping ", length(big), " events with > ",
                             max_size, " actors")
    d <- d[!ev %in% big]
  }
  d[, `:=`(ev = as.factor(ev), ac = as.factor(ac))]
  B <- sparseMatrix(i = as.integer(d$ev), j = as.integer(d$ac), x = 1,
                    dims = c(nlevels(d$ev), nlevels(d$ac)),
                    dimnames = list(levels(d$ev), levels(d$ac)))
  A <- Matrix::crossprod(B, B)                 # actor x actor
  n_ev <- diag(A)                              # events per actor
  diag(A) <- 0
  A <- Matrix::drop0(A)
  tri <- Matrix::summary(Matrix::triu(A))      # upper triangle -> edge list
  edges <- data.table(from = colnames(A)[tri$i],
                      to   = colnames(A)[tri$j],
                      weight = tri$x)
  list(edges = edges,
       nodes = data.table(name = colnames(A), n_events = as.numeric(n_ev)),
       incidence = B)
}

## HELPER 2 - assemble an igraph object with node attributes attached
make_net <- function(proj, node_attr = NULL, by = "name") {
  nodes <- proj$nodes
  if (!is.null(node_attr)) {
    node_attr <- copy(as.data.table(node_attr))
    node_attr[, (by) := as.character(get(by))]   # node names are always character
    node_attr <- unique(node_attr, by = by)
    nodes <- merge(nodes, node_attr, by.x = "name", by.y = by, all.x = TRUE)
  }
  graph_from_data_frame(proj$edges, directed = FALSE, vertices = nodes)
}


## ---------------------------------------------------------------------------
## HELPER 4 - Burt's effective size (igraph has constraint(), not this one)
## ---------------------------------------------------------------------------
effective_size <- function(g) {
  A <- as_adjacency_matrix(g, sparse = TRUE); A <- (A > 0) * 1
  deg <- Matrix::rowSums(A)
  redundancy <- Matrix::rowSums((A %*% A) * A)      # 2 x ties among my contacts
  as.numeric(deg - redundancy / pmax(deg, 1))
}

## ---------------------------------------------------------------------------
## HELPER 5 - Gould & Fernandez (1989) brokerage roles
## ---------------------------------------------------------------------------
## v brokers the 2-path i -> v -> j when i and j are NOT directly tied. Given a
## group partition, the role depends on where i, v and j sit:
##   coordinator  i, v, j same group      gatekeeper      i outside, v, j inside
##   representative  i, v inside, j out   consultant      i, j in one other group
##   liaison      i, v, j all different
## In an UNDIRECTED network gatekeeper == representative by construction.
## Block 6 discusses how to read the output; the cost grows with degree^2, so
## filter the network first.
brokerage_roles <- function(g, group) {
  A <- as_adjacency_matrix(g, sparse = TRUE); A <- (A > 0) * 1
  if (!is_directed(g)) A <- ((A + Matrix::t(A)) > 0) * 1
  grp <- factor(group); k <- nlevels(grp); gi <- as.integer(grp); n <- vcount(g)
  nm <- if (is.null(V(g)$name)) as.character(seq_len(n)) else V(g)$name
  out <- matrix(0, n, 5, dimnames = list(nm,
    c("coordinator", "gatekeeper", "representative", "consultant", "liaison")))
  for (v in seq_len(n)) {
    I <- which(A[, v] > 0); O <- which(A[v, ] > 0)
    if (!length(I) || !length(O)) next
    M <- outer(tabulate(gi[I], k), tabulate(gi[O], k))     # all (g_i, g_j) pairs
    both <- intersect(I, O)                                 # drop i == j
    if (length(both)) diag(M) <- diag(M) - tabulate(gi[both], k)
    sub <- A[I, O, drop = FALSE]                            # drop direct i -> j
    if (sum(sub)) {
      GI <- sparseMatrix(seq_along(I), gi[I], dims = c(length(I), k))
      GO <- sparseMatrix(seq_along(O), gi[O], dims = c(length(O), k))
      M <- M - as.matrix(Matrix::t(GI) %*% sub %*% GO)
    }
    gv <- gi[v]; own <- rep(FALSE, k); own[gv] <- TRUE
    other <- M[!own, !own, drop = FALSE]
    out[v, ] <- c(M[gv, gv], sum(M[!own, gv]), sum(M[gv, !own]),
                  sum(diag(other)), sum(other) - sum(diag(other)))
  }
  as.data.table(out, keep.rownames = "name")
}

## ---------------------------------------------------------------------------
## HELPER 6 - economic / knowledge complexity (Hidalgo & Hausmann 2009)
## ---------------------------------------------------------------------------
## Input: a binary ACTOR x CATEGORY matrix M (countries x products, regions x
## technologies, ...). Returns the two complexity indices, i.e. the second
## eigenvectors of the two "method of reflections" operators:
##      Mcc = D^-1 M U^-1 M'      (actor side, ECI/KCI)
##      Mpp = U^-1 M' D^-1 M      (category side, PCI/TCI)
## Signs are the whole difficulty. Conventions used here:
##   - actor index increases with DIVERSITY (diversified actors are complex);
##   - category index is aligned with the actor index (a complex category is one
##     that only complex actors have) - equivalently it DECREASES with ubiquity.
## Getting this backwards silently returns the ranking upside down, which is the
## most common mistake with these measures: always sanity-check the extremes.
complexity <- function(M) {
  d <- rowSums(M); u <- colSums(M)
  M <- M[d > 0, u > 0, drop = FALSE]; d <- rowSums(M); u <- colSums(M)
  ev2 <- function(A) as.numeric(scale(Re(eigen(A)$vectors[, 2])))
  aci <- ev2((M / d) %*% t(sweep(M, 2, u, "/")))          # actor side
  if (cor(aci, d) < 0) aci <- -aci                        # diversified = complex
  cci <- ev2(t(sweep(M, 2, u, "/")) %*% (M / d))          # category side
  avg_actor <- as.numeric(t(M) %*% aci / u)               # mean ACI of holders
  if (cor(cci, avg_actor) < 0) cci <- -cci
  list(actor = setNames(aci, rownames(M)),
       category = setNames(cci, colnames(M)),
       diversity = d, ubiquity = u)
}

## HELPER 7 - the giant (largest) component, we often work on it
giant <- function(g) {
  cmp <- components(g)
  induced_subgraph(g, V(g)[cmp$membership == which.max(cmp$csize)])
}

cat("Setup complete -", R.version.string, "| igraph", as.character(packageVersion("igraph")), "\n")

---

## BLOCK 1 of the session (~18 min) - PATENT DATA: the co-inventor network

Data: OECD REGPAT (May 2025), EPO applications, inventor-region file, joined
with CPC codes. Extract used here: all EPO patent applications with priority
year 2010-2019 that (i) have at least one Italian inventor and (ii) carry at
least one "green" CPC code (Y02/Y04S climate-change mitigation tagging).
What we do: from a raw patent-inventor table to (a) a co-invention network,
(b) inventor-level positional measures, (c) region-level indicators you can
put in a regression.


### 1. The raw data: one row = one inventor on one patent


In [ ]:
inv <- fread(daisy_data("pat_green_inventors_IT.csv.gz"))
inv
str(inv)

## appln_id  : patent application (the "event")
## person_id : REGPAT disambiguated inventor id (the "actor")
## reg_code  : NUTS-3 region of residence of the inventor, ctry_code: country
## prio_year : priority year = closest to the moment of invention

## How much data do we have?
inv[, .(patents = uniqueN(appln_id), inventors = uniqueN(person_id))]

## Patents and inventors per year
by_year <- inv[, .(patents = uniqueN(appln_id), inventors = uniqueN(person_id)),
               by = prio_year][order(prio_year)]
by_year

ggplot(by_year, aes(prio_year, patents)) +
  geom_col(fill = "grey40") +
  labs(x = NULL, y = "green EPO applications with IT inventors") +
  theme_minimal()

## Team size: this is what drives the density of the co-invention network
team <- inv[, .(size = uniqueN(person_id)), by = appln_id]
team[, .(mean = mean(size), median = as.double(median(size)), max = max(size))]
table(team$size)

## Careful: many patents have ONE inventor -> they will be isolated nodes
## Careful #2: inventors can be counted in several regions (reg_share) and
## patents in several technologies; use fractional counts when you aggregate.

## Where are the inventors? (NUTS-3 -> NUTS-2 by truncation)
inv[, nuts2 := substr(reg_code, 1, 4)]
inv[ctry_code == "IT", .(inventors = uniqueN(person_id)), by = nuts2][order(-inventors)][1:10]

### 2. From the two-mode (patent x inventor) to the co-invention network


In [ ]:
## The tie is *not observed*: we infer it from co-participation in a patent.
pr <- proj_two_mode(inv, event = "appln_id", actor = "person_id")

head(pr$edges)          # weight = number of patents co-invented
head(pr$nodes)          # n_events = number of patents of the inventor

## WARNING - the projection turns every team into a clique. The biggest patent
## in this sample has 68 inventors: that single document produces 68*67/2 =
## 2,278 ties. Whether to keep such documents is a research design choice.
team[order(-size)][1:5]
pr20 <- proj_two_mode(inv, "appln_id", "person_id", max_size = 20)
c(ties_all = nrow(pr$edges), ties_teams_below_20 = nrow(pr20$edges))

## Attach inventor attributes (one row per inventor: first region observed)
attr_inv <- inv[, .(inv_name = inv_name[1], ctry = ctry_code[1],
                    nuts2 = nuts2[1], reg = reg_code[1],
                    first_year = min(prio_year)), by = person_id]

g <- make_net(pr, node_attr = attr_inv, by = "person_id")
g

## Basic anatomy of the network
vcount(g); ecount(g)
edge_density(g)
mean(degree(g))
table(degree(g) == 0)                 # isolates = single-inventor patents only

## Components: co-invention networks are always highly fragmented
cmp <- components(g)
cmp$no                                # number of components
sort(cmp$csize, decreasing = TRUE)[1:10]
max(cmp$csize) / vcount(g)            # share of inventors in the giant component

gc_net <- giant(g)
gc_net

## Small-world-ness of the giant component
mean_distance(gc_net)
diameter(gc_net, weights = NA)
transitivity(gc_net, type = "global")   # very high: teams are cliques by construction
## benchmark against a random graph of the same size and density
rnd <- sample_gnm(vcount(gc_net), ecount(gc_net))
c(observed = transitivity(gc_net, type = "global"),
  random   = transitivity(rnd, type = "global"))

## Degree distribution: fat tailed, as usual in collaboration networks
ggplot(data.table(k = degree(g)), aes(k)) +
  geom_histogram(binwidth = 1, fill = "grey40") +
  scale_y_log10() + labs(x = "degree (number of distinct co-inventors)") +
  theme_minimal()

### 3. Positions: who matters, and in which sense?


In [ ]:
V(g)$degree      <- degree(g)
V(g)$strength    <- strength(g)                       # weighted by n. of patents
V(g)$betw        <- betweenness(g, weights = NA, normalized = TRUE)
V(g)$eigen       <- eigen_centrality(g, weights = NA)$vector
V(g)$constraint  <- constraint(g)                     # Burt: LOW = structural holes
V(g)$clust       <- transitivity(g, type = "local", isolates = "zero")

cent <- as.data.table(as_data_frame(g, what = "vertices"))
setnames(cent, "name", "person_id")

## Top inventors by different criteria - they are NOT the same people
cent[order(-degree)][1:10, .(inv_name, nuts2, n_events, degree, strength, betw)]
cent[order(-betw)][1:10,   .(inv_name, nuts2, n_events, degree, betw, constraint)]

## Are patent counts and network position the same information?
cent[n_events > 0, round(cor(.SD, use = "pairwise"), 2),
     .SDcols = c("n_events", "degree", "strength", "betw", "eigen", "constraint")]

## => centrality is correlated with productivity but far from collinear: this is
##    why network position enters innovation regressions on its own.

### 4. Visualise (when not fundamental only plot a subgraph you can actually read)


In [ ]:
sub <- giant(g)
sub <- induced_subgraph(sub, V(sub)[degree(sub) > 1])

ggraph(sub, layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey75") +
  scale_edge_width(range = c(0.2, 1.5)) +
  geom_node_point(aes(size = degree, fill = ctry), shape = 21, colour = "white") +
  scale_size(range = c(1, 6)) +
  theme_graph(base_family = "sans") +
  labs(title = "Giant component, green co-invention network (IT, 2010-2019)",
       fill = "inventor country")

### 5. From nodes to variables: region-level network indicators


In [ ]:
## This is what usually ends up in an econometric model: aggregate inventor
## positions by region (or firm, or year window) and use them as regressors.

## Or.. Directly compute the region-level network and indicators

reg_ind <- cent[ctry == "IT" & nuts2 != "", .(
  inventors        = .N,
  patents          = sum(n_events),
  avg_degree       = mean(degree),
  avg_strength     = mean(strength),
  avg_betweenness  = mean(betw),
  avg_constraint   = mean(constraint, na.rm = TRUE),
  share_connected  = mean(degree > 0),
  top_inventor     = inv_name[which.max(degree)]
), by = nuts2][order(-patents)]
reg_ind[1:15]


## Two routes to a regional indicator - and they are not the same object.
## (A) above: build the INVENTOR network, then average positions by region.
## (B) below: aggregate the actors first, and build the network directly BETWEEN
##     NUTS-2 regions. The event is still the patent; the actor is now the region.
##     A tie means "inventors of these two regions signed the same patent", and
##     its weight counts those patents.
pr_reg <- proj_two_mode(inv[nuts2 != ""], event = "appln_id", actor = "nuts2")

reg_attr <- unique(inv[nuts2 != "", .(nuts2, ctry = ctry_code)], by = "nuts2")
g_reg <- make_net(pr_reg, node_attr = reg_attr, by = "nuts2")
g_reg
## n_events is now the number of green patents of the region (a size variable),
## and the network is small enough to look at as a whole
c(regions = vcount(g_reg), ties = ecount(g_reg),
  density = round(edge_density(g_reg), 3),
  giant_share = round(max(components(g_reg)$csize) / vcount(g_reg), 2))

## Careful: patents with all inventors in ONE region produce no tie at all (the
## projection drops the diagonal). Co-invention *within* a region is invisible
## here - if it matters for your question, keep it as a separate variable:
within_reg <- inv[nuts2 != "", .(n_reg = uniqueN(nuts2)), by = appln_id]
within_reg[, .(patents = .N,
               single_region_share = round(mean(n_reg == 1), 2))]

## Region-level positions, computed on the region network itself
V(g_reg)$degree   <- degree(g_reg)             # n. of partner regions
V(g_reg)$strength <- strength(g_reg)           # n. of co-patents with them
V(g_reg)$betw     <- betweenness(g_reg, weights = NA, normalized = TRUE)
V(g_reg)$constr   <- constraint(g_reg)

reg_net <- as.data.table(as_data_frame(g_reg, what = "vertices"))
setnames(reg_net, c("name", "n_events"), c("nuts2", "patents_reg"))
reg_net[ctry == "IT"][order(-strength)][1:10,
        .(nuts2, patents_reg = round(patents_reg), degree, strength,
          betw = round(betw, 3), constr = round(constr, 2))]

## Do the two routes give the same ranking? (they measure different things)
comp_route <- merge(reg_ind[, .(nuts2, patents, avg_degree, avg_constraint)],
                    reg_net[, .(nuts2, degree, strength, constr)], by = "nuts2")
round(cor(comp_route[, -1], method = "spearman"), 2)
## An inventor-level average says "how connected are our inventors";
## the region network says "how connected is our region to other regions".
## Ecological fallacy runs in both directions - state which one you mean.

## The Italian part of the region network, drawn
g_it <- induced_subgraph(g_reg, V(g_reg)[ctry == "IT"])
g_it <- delete_edges(g_it, E(g_it)[weight < 2])
g_it <- induced_subgraph(g_it, V(g_it)[degree(g_it) > 0])

ggraph(g_it, layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey70") +
  scale_edge_width(range = c(0.2, 2.5)) +
  geom_node_point(aes(size = n_events), fill = "#2c7fb8", shape = 21, colour = "white") +
  geom_node_text(aes(label = name), size = 3, repel = TRUE) +
  scale_size(range = c(2, 12)) +
  theme_graph(base_family = "sans") + theme(legend.position = "none") +
  labs(title = "Green co-invention between Italian NUTS-2 regions, 2010-2019",
       subtitle = "ties with at least 2 shared patents; node size = regional patents")

## Cross-border openness: share of an inventor's ties that go outside the region
el <- as.data.table(as_data_frame(g, what = "edges"))
nuts_of <- setNames(V(g)$nuts2, V(g)$name)
el[, `:=`(n_from = nuts_of[from], n_to = nuts_of[to])]
ext <- rbind(el[, .(nuts2 = n_from, ext = as.integer(n_from != n_to), weight)],
             el[, .(nuts2 = n_to,   ext = as.integer(n_from != n_to), weight)])
open_reg <- ext[, .(external_tie_share = weighted.mean(ext, weight)), by = nuts2]
reg_ind <- merge(reg_ind, open_reg, by = "nuts2", all.x = TRUE)
reg_ind[order(-patents)][1:10, .(nuts2, patents, avg_degree, avg_constraint,
                                 share_connected, external_tie_share)]

fwrite(reg_ind, "output_region_network_indicators.csv")

### 6. The network boundary is a decision, not a fact


In [ ]:
## A small function that runs the whole pipeline and returns summary statistics
net_stats <- function(dt) {
  p <- proj_two_mode(dt, "appln_id", "person_id")
  gg <- make_net(p)
  cmp <- components(gg)
  data.table(inventors = vcount(gg), ties = ecount(gg),
             density = edge_density(gg),
             avg_degree = mean(degree(gg)),
             giant_share = max(cmp$csize) / vcount(gg),
             clustering = transitivity(gg, type = "global"))
}

## So far a tie exists only if two inventors share a GREEN patent. But the same
## inventors also collaborate on non-green patents. Same actors, wider boundary:
inv_all <- fread(daisy_data("pat_all_inventors_ITgreen.csv.gz"))
inv_all[, .(patents = uniqueN(appln_id), green = uniqueN(appln_id[green == 1]))]

boundary <- rbind(
  `green ties only` = net_stats(inv),
  `all co-patenting ties` = net_stats(inv_all), idcol = "boundary")
boundary
## Connectivity, average degree and the giant component all move: any statement
## about "the" position of an inventor is conditional on this choice.

## ... and so is the *population* boundary. Same pipeline, run on the
## full REGPAT for 17 countries (green patents, 2015-2019):
bench <- fread(daisy_data("green_coinvention_country_benchmark.csv"))
bench[order(-patents)]
## Note (i) how small the giant component is everywhere over a 5-year window,
## (ii) FI: avg_degree of 21 driven by a handful of very large teams - always
## look for the mega-document before interpreting a "dense" network.

### 7. IF WE HAVE TIME - does the network change over time?


In [ ]:
## We reuse net_stats() defined above on moving windows.
windows <- list(`2010-2013` = 2010:2013, `2014-2016` = 2014:2016,
                `2017-2019` = 2017:2019)
evo <- rbindlist(lapply(windows, function(y) net_stats(inv[prio_year %in% y])),
                 idcol = "window")
evo

## Discussion: fragmentation, densification, and the sensitivity of ALL of this
## to the length of the time window - a modelling choice, not a data property.

---

## BLOCK 2 of the session (~16 min) - CORDIS: EU-funded collaborative projects

Data: CORDIS Horizon Europe, open data (CC-BY), release 2026-08-06,
https://cordis.europa.eu/dataset  (project.csv, organization.csv,
policyPriorities.csv, euroSciVoc.csv ...)
Extract used here: the 4,604 HE projects tagged as 100% climate-relevant or
classified under a sustainability-related euroSciVoc term, and their 44,644
participations (16,116 distinct organisations).
Why this source: unlike patents and papers, here the collaboration is a
*contractual, funded and dated* relationship, with money attached to each
partner - and it covers the actors that patents miss (universities, public
bodies, NGOs, SMEs).


### 1. The data


In [ ]:
part <- fread(daisy_data("cordis_he_participants.csv.gz"))
proj <- fread(daisy_data("cordis_he_projects.csv"))

part          # one row = one organisation in one project
proj[1:3, .(project_id, acronym, start_year, programme, ec_contrib)]

part[, .(projects = uniqueN(project_id), organisations = uniqueN(org_id),
         participations = .N)]

## Consortium size: the "event size" that will drive the projection
size <- part[, .(partners = .N), by = project_id]
size[, .(mean = mean(partners), median = as.double(median(partners)), max = max(partners))]
size[order(-partners)][1:5]
proj[project_id %in% size[order(-partners)][1:3]$project_id, .(acronym, title)]

## Who participates? (HES = higher education, REC = research org, PRC = private,
## PUB = public body, OTH = other)
part[, .(participations = .N, orgs = uniqueN(org_id),
         ec_meur = round(sum(ec_contrib_org, na.rm = TRUE) / 1e6)),
     by = activity_type][order(-participations)]

## Top countries by participation and by money
part[, .(participations = .N,
         ec_meur = round(sum(ec_contrib_org, na.rm = TRUE) / 1e6)),
     by = country][order(-ec_meur)][1:15]

## Projects per year
part[, .(projects = uniqueN(project_id)), by = start_year][order(start_year)]

### 2. The organisation collaboration network


In [ ]:
## Same helper as for patents: the event is the project, the actor the partner.
## Mega-consortia (60+ partners) would dominate the topology, so we look at both.
pr_all <- proj_two_mode(part, "project_id", "org_id")
pr_lim <- proj_two_mode(part, "project_id", "org_id", max_size = 40)
c(ties_all = nrow(pr_all$edges), ties_below_40_partners = nrow(pr_lim$edges))

org_attr <- part[, .(org_name = org_name[1], country = country[1],
                     nuts = nuts[1], type = activity_type[1], sme = sme[1],
                     eur = sum(ec_contrib_org, na.rm = TRUE)), by = org_id]

g <- make_net(pr_lim, node_attr = org_attr, by = "org_id")
g

## Anatomy: EU funding networks look nothing like co-invention networks
cmp <- components(g)
c(nodes = vcount(g), edges = ecount(g), density = edge_density(g),
  components = cmp$no, giant_share = max(cmp$csize) / vcount(g),
  clustering = transitivity(g, type = "global"),
  avg_degree = mean(degree(g)))

## a connected, dense, high-clustering core: the "policy-made" network

### 3. Who is central - and does centrality mean the same as money?


In [ ]:
V(g)$degree   <- degree(g)                       # distinct partners
V(g)$strength <- strength(g)                     # partner-projects
V(g)$betw     <- betweenness(g, weights = NA, normalized = TRUE)
V(g)$eigen    <- eigen_centrality(g, weights = NA)$vector
V(g)$constr   <- constraint(g)

nodes <- as.data.table(as_data_frame(g, what = "vertices"))
nodes[order(-degree)][1:15, .(org_name, country, type, n_events, degree, betw,
                              eur_meur = round(eur / 1e6, 1))]

## Money and network position are related but not the same thing
nodes[, round(cor(cbind(n_events, degree, strength, betw, eigen, eur),
                  use = "pairwise"), 2)]

## Brokers vs hubs: rank difference tells you who bridges rather than accumulates
nodes[, `:=`(r_deg = frankv(-degree), r_betw = frankv(-betw))]
nodes[degree > 20][order(r_betw - r_deg)][1:10,
      .(org_name, country, type, degree, betw = round(betw, 4))]

### 4. Is EU research integrated, or nationally clustered?


In [ ]:
## (a) assortativity: do organisations partner with similar organisations?
assortativity_nominal(g, factor(V(g)$country))     # by country
assortativity_nominal(g, factor(V(g)$type))        # by type of organisation
assortativity_degree(g)                            # hubs with hubs?

## (b) Share of ties that cross national borders (compare with the
##     0.26-0.46 range we found for regions in the co-invention network)
el0 <- as.data.table(as_data_frame(g, what = "edges"))
ctry_of <- setNames(V(g)$country, V(g)$name)
el0[, cross := as.integer(ctry_of[from] != ctry_of[to])]
el0[, .(cross_border_share = weighted.mean(cross, weight))]

## (c) communities in the giant component and their national composition
##     (which algorithm? which resolution? how stable? -> 06_brokerage_communities.R,
##      which also computes Gould-Fernandez brokerage roles on this same network)
gg <- giant(g)
comm <- cluster_louvain(gg, weights = E(gg)$weight)
length(comm); sort(sizes(comm), decreasing = TRUE)[1:8]

memb <- data.table(org_id = V(gg)$name, country = V(gg)$country,
                   type = V(gg)$type, comm = membership(comm))
top_comm <- memb[, .N, by = comm][order(-N)][1:6]$comm

## how concentrated is each community by country? (HHI = 1 -> single country)
memb[comm %in% top_comm, .(
  orgs = .N,
  top_country = names(which.max(table(country))),
  top_share = round(max(prop.table(table(country))), 2),
  hhi = round(sum(prop.table(table(country))^2), 3)), by = comm][order(-orgs)]

## benchmark: concentration of the whole network
memb[, .(hhi_all = round(sum(prop.table(table(country))^2), 3))]
## => communities are thematic-institutional, not national: the opposite of what
##    we found for co-invention. Worth a slide in any paper on EU integration.

### 5. Aggregate the same data at NUTS-2 level (and plot it)


In [ ]:
## The projection helper works at any level of aggregation: just change "actor".
## Regions are the level at which most of the innovation-policy literature works,
## and CORDIS geocodes every participant to a NUTS code.

## First look at what the geography column actually contains
part[, .N, by = .(nuts_length = nchar(nuts))][order(-N)]
## 5 characters = NUTS-3, 2 = country only (non-EU partners), a handful of odd
## ones. NUTS exists only for Europe: aggregating to regions silently DROPS
## every third-country partner. That is a change of population, not a detail.
part[, nuts2 := ifelse(nchar(nuts) >= 4, substr(nuts, 1, 4), NA_character_)]
part[, .(participations = .N, with_region = sum(!is.na(nuts2)),
         share_kept = round(mean(!is.na(nuts2)), 3))]

pr_reg  <- proj_two_mode(part[!is.na(nuts2)], "project_id", "nuts2")
reg_att <- unique(part[!is.na(nuts2), .(nuts2, country)], by = "nuts2")
g_reg   <- make_net(pr_reg, node_attr = reg_att, by = "nuts2")
g_reg

c(regions = vcount(g_reg), ties = ecount(g_reg),
  density = round(edge_density(g_reg), 3),
  giant_share = round(max(components(g_reg)$csize) / vcount(g_reg), 3))

## Which regions sit at the centre of EU climate research?
V(g_reg)$degree   <- degree(g_reg)
V(g_reg)$strength <- strength(g_reg)
V(g_reg)$betw     <- betweenness(g_reg, weights = NA, normalized = TRUE)
reg_nodes <- as.data.table(as_data_frame(g_reg, what = "vertices"))
setnames(reg_nodes, c("name", "n_events"), c("nuts2", "projects"))
reg_nodes[order(-strength)][1:12, .(nuts2, country, projects, degree, strength,
                                    betw = round(betw, 3))]

## Is regional collaboration national or European? (compare with the 26-46%
## extra-regional share of the co-invention network in block 1)
el_r <- as.data.table(as_data_frame(g_reg, what = "edges"))
ctry_of_reg <- setNames(V(g_reg)$country, V(g_reg)$name)
el_r[, cross := as.integer(ctry_of_reg[from] != ctry_of_reg[to])]
el_r[, .(cross_country_share = round(weighted.mean(cross, weight), 3))]

## Region-level indicators: raw weights favour big regions, so normalise.
## A simple revealed-collaboration index: observed ties / expected under
## independence given each region's number of participations.
tot <- data.table(nuts2 = V(g_reg)$name, part = V(g_reg)$n_events)
el_r <- merge(merge(el_r, tot, by.x = "from", by.y = "nuts2"),
              tot, by.x = "to", by.y = "nuts2", suffixes = c("_f", "_t"))
el_r[, rci := weight / (part_f * part_t / sum(tot$part))]
el_r[weight >= 10][order(-rci)][1:10, .(from, to, weight, rci = round(rci, 1),
                                        cross)]
## The strongest *relative* ties are pairs of regions in the same country, or in
## neighbouring ones: geography and institutional proximity survive even inside
## a supranational programme designed to overcome them.

## Draw the backbone: 323 regions are too many, keep the strongest ties
g_plot <- delete_edges(g_reg, E(g_reg)[weight < 40])
g_plot <- induced_subgraph(g_plot, V(g_plot)[degree(g_plot) > 0])
V(g_plot)$projects <- V(g_plot)$n_events

ggraph(g_plot, layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey80", edge_alpha = .8) +
  scale_edge_width(range = c(0.1, 2.5)) +
  geom_node_point(aes(size = projects, fill = country), shape = 21, colour = "white") +
  geom_node_text(aes(label = name), size = 2.6, repel = TRUE, max.overlaps = 20) +
  scale_size(range = c(2, 11)) +
  theme_graph(base_family = "sans") + theme(legend.position = "none") +
  labs(title = "NUTS-2 co-participation, Horizon Europe climate projects",
       subtitle = "ties with at least 40 shared projects; node size = participations")

## Exercise for later: the same three lines with actor = "country" give the
## country network. Compare the two rankings - Ile-de-France against France.

### 6. IF WE HAVE TIME - tie formation: new or repeated partners?


In [ ]:
## A dynamic indicator you can build from any project database: how much of the
## collaboration in t is with partners already met before t?
early <- proj_two_mode(part[start_year <= 2022], "project_id", "org_id")$edges
late  <- proj_two_mode(part[start_year >= 2024], "project_id", "org_id")$edges
key   <- function(d) paste(pmin(d$from, d$to), pmax(d$from, d$to))
mean(key(late) %in% key(early))       # share of repeated ties
## Repetition rate = trust/lock-in vs renewal of the consortium ecosystem.

### 7. And the same data as a *topic* network


In [ ]:
## euroSciVoc classifies every project into scientific fields, so the SAME
## projection turns projects into a map of what EU climate research is about.
## We build that map in 04_indicators.R, next to the patent knowledge space and
## the product space: it is the same construction applied to three different
## category systems (technologies, topics, products).

---

## BLOCK 3 of the session (~8 min) - PUBLICATION DATA: the OpenAlex API

Publications get a session of their own in this school, so here we only look
at (i) how to pull relational data out of the OpenAlex API in three lines,
(ii) how the very same projection logic gives co-authorship networks, and
(iii) what changes when the actors are institutions rather than people.
OpenAlex: fully open (CC0), no key, ~250M works. Be polite: add your e-mail
("polite pool"), max 10 requests/second, 100k/day.
R packages worth knowing: openalexR (wrapper), rcrossref, europepmc,
bibliometrix. Here we use the raw API so you see what happens.


In [ ]:

MAIL <- "your.name@your.university.it"      # <- put YOUR address here
oa <- function(path, ...) {
  url <- paste0("https://api.openalex.org/", path, "&mailto=", MAIL)
  fromJSON(URLencode(url), simplifyVector = FALSE)
}

### 1. Aggregate queries: indicators without downloading any record


In [ ]:
## group_by returns counts, not records: perfect for descriptive statistics.
res <- oa(paste0("works?filter=title_and_abstract.search:circular economy,",
                 "publication_year:2015-2024&group_by=publication_year"))
by_year <- rbindlist(lapply(res$group_by, function(x)
  data.table(year = as.integer(x$key), works = x$count)))[order(year)]
by_year

## Which countries publish on it?
res <- oa(paste0("works?filter=title_and_abstract.search:circular economy,",
                 "publication_year:2020-2024&group_by=authorships.countries"))
rbindlist(lapply(res$group_by, function(x)
  data.table(country = x$key_display_name, works = x$count)))[1:15]

### 2. Record-level download: works with their authorships


In [ ]:
## 200 records per page, cursor paging. select= keeps the payload small.
fetch_works <- function(n_pages = 2) {
  q <- paste0("works?filter=title_and_abstract.search:circular economy,",
              "publication_year:2020-2024,type:article,has_orcid:true",
              "&select=id,display_name,publication_year,cited_by_count,authorships",
              "&per-page=200")
  out <- list(); cursor <- "*"
  for (i in seq_len(n_pages)) {
    r <- oa(paste0(q, "&cursor=", cursor))
    out[[i]] <- r$results; cursor <- r$meta$next_cursor
    if (is.null(cursor)) break
  }
  unlist(out, recursive = FALSE)
}

## flatten works x authors x institutions into a long table
flatten_authorships <- function(works) rbindlist(lapply(works, function(w)
  rbindlist(lapply(w$authorships, function(a) {
    ins <- if (length(a$institutions)) a$institutions else list(list())
    rbindlist(lapply(ins, function(s) data.table(
      work_id      = sub(".*/", "", w$id),
      year         = w$publication_year,
      cited_by     = w$cited_by_count,
      author_id    = sub(".*/", "", a$author$id %||% NA_character_),
      author_name  = a$author$display_name %||% NA_character_,
      inst_id      = sub(".*/", "", s$id %||% NA_character_),
      inst_name    = s$display_name %||% NA_character_,
      inst_country = s$country_code %||% NA_character_,
      inst_type    = s$type %||% NA_character_)))
  }), fill = TRUE)), fill = TRUE)

## live if the wifi cooperates, otherwise the cached extract (2,000 works)
aut <- tryCatch(flatten_authorships(fetch_works(2)),
                error = function(e) {
                  message("API unreachable, using the cached extract")
                  fread(daisy_data("openalex_ce_authorships.csv.gz"))
                })
if (nrow(aut) < 1000) aut <- fread(daisy_data("openalex_ce_authorships.csv.gz"))

aut[, .(works = uniqueN(work_id), authors = uniqueN(author_id),
        institutions = uniqueN(inst_id), countries = uniqueN(inst_country))]

### 3. Three networks out of one table (same helper as blocks 1 and 2)


In [ ]:
## (a) co-authorship between researchers
aut_attr <- unique(aut[, .(author_id, author_name, country = inst_country)],
                   by = "author_id")
g_aut <- make_net(proj_two_mode(aut, "work_id", "author_id", max_size = 30),
                  node_attr = aut_attr, by = "author_id")
cmp <- components(g_aut)
c(authors = vcount(g_aut), ties = ecount(g_aut),
  giant_share = max(cmp$csize) / vcount(g_aut))

## (b) collaboration between institutions
g_ins <- make_net(proj_two_mode(aut, "work_id", "inst_id", max_size = 30),
                  node_attr = unique(aut[!is.na(inst_id),
                                         .(inst_id, inst_name, inst_country, inst_type)]),
                  by = "inst_id")
g_ins
V(g_ins)$degree <- degree(g_ins)
V(g_ins)$betw   <- betweenness(g_ins, weights = NA, normalized = TRUE)
ins <- as.data.table(as_data_frame(g_ins, what = "vertices"))
ins[order(-degree)][1:12, .(inst_name, inst_country, inst_type,
                            papers = n_events, degree, betw = round(betw, 3))]

## (c) country co-publication network
g_ctry <- make_net(proj_two_mode(unique(aut[, .(work_id, inst_country)]),
                                 "work_id", "inst_country"))
sort(strength(g_ctry), decreasing = TRUE)[1:12]

el <- as.data.table(as_data_frame(g_ctry, what = "edges"))
el[order(-weight)][1:10]

ggraph(delete_edges(g_ctry, E(g_ctry)[weight < 5]), layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey80") +
  scale_edge_width(range = c(0.2, 3)) +
  geom_node_point(aes(size = n_events), fill = "#d95f0e", shape = 21, colour = "white") +
  geom_node_text(aes(label = name), size = 3, repel = TRUE) +
  theme_graph(base_family = "sans") + theme(legend.position = "none") +
  labs(title = "Country co-publication network, circular economy research")

### 4. What to keep in mind (and what connects this to the other blocks)


In [ ]:
## - Author disambiguation: OpenAlex ids are algorithmic. Same problem as
##   REGPAT person_id and CORDIS organisationID: measurement error in the NODES
##   propagates to every network statistic. Check your top-degree actors by hand.
## - Coverage/selection: our query is a keyword search; a different query is a
##   different network. Prefer topic/concept ids or a validated keyword list, and
##   always report the query in the paper.
## - Linking science and technology: patent front-page and non-patent-literature
##   citations (PATSTAT TLS214, Lens.org, Reliance-on-Science) let you build
##   *directed* science -> technology networks. That is where publication and
##   patent data meet.

---

## BLOCK 4 of the session (~14 min) - TRADE AND GLOBAL VALUE CHAINS:

observed, directed, valued networks
Everything so far was an *inferred* tie: two actors shared a document. Trade
data are the opposite case, and they break most of our habits:
- the tie is OBSERVED and has a VALUE (millions of USD), not a count;
- it is DIRECTED (i sells to j is not j sells to i);
- the network is essentially COMPLETE: almost every pair trades something.
Density carries no information; the whole signal is in the weights.
Filtering stops being cosmetic and becomes part of the method.
Data: OECD TiVA (FDVA table) - value added of country i embodied in the final
demand of country j, 80 economies, 1995/2005/2015/2022, total and by industry.
Caveat to state out loud: TiVA is not a measurement, it is the output of an
inter-country input-output model. The network inherits its assumptions.


### 1. Value added flows between countries


In [ ]:
va  <- fread(daisy_data("tiva_va_bilateral.csv.gz"))
reg <- fread(daisy_data("country_regions.csv"))
va

## the diagonal is domestic value added in domestic final demand: not a tie
va <- va[source != destination]
va[year == 2022, .(flows = .N, total_bn = round(sum(va_musd) / 1000))]
va[year == 2022][order(-va_musd)][1:8]

## how much of world trade in value added is a handful of pairs?
va[year == 2022, .(top10_share = round(100 * sum(sort(va_musd, decreasing = TRUE)[1:10]) /
                                        sum(va_musd)))]

### 2. A directed, weighted graph


In [ ]:
g22 <- graph_from_data_frame(va[year == 2022, .(source, destination, weight = va_musd)],
                             directed = TRUE,
                             vertices = reg[, .(name = iso3, region)])
g22
c(nodes = vcount(g22), edges = ecount(g22), density = round(edge_density(g22), 3))
## density ~ 1: the topology is a complete digraph, so degree is useless here...

## ... and strength is everything
V(g22)$out_str <- strength(g22, mode = "out")      # value added SOLD abroad
V(g22)$in_str  <- strength(g22, mode = "in")       # foreign VA ABSORBED
nodes <- as.data.table(as_data_frame(g22, what = "vertices"))
nodes[, `:=`(net = in_str - out_str,
             share_out = round(100 * out_str / sum(out_str), 1))]
nodes[order(-out_str)][1:12, .(name, region, out_bn = round(out_str/1000),
                               in_bn = round(in_str/1000), share_out)]

## Concentration: is a country's foreign demand diversified or hostage to one market?
el22 <- as.data.table(as_data_frame(g22, what = "edges"))
hhi <- el22[, .(hhi = sum((weight / sum(weight))^2),
                top_market = to[which.max(weight)],
                top_share = round(100 * max(weight) / sum(weight))), by = from]
hhi[order(-hhi)][1:10]
hhi[from %in% c("DEU","ITA","CHN","USA","MEX","IRL")]
## Mexico and Canada are structurally exposed to one market; Germany and Italy
## are not. Same network, a country-level risk indicator.

## Dyadic asymmetry: who is upstream of whom
dy <- merge(el22, el22, by.x = c("from","to"), by.y = c("to","from"))
dy <- dy[from < to, .(a = from, b = to, a_to_b = weight.x, b_to_a = weight.y)]
dy[, imbalance := round((a_to_b - b_to_a) / (a_to_b + b_to_a), 2)]
dy[a_to_b + b_to_a > 50000][order(-imbalance)][1:8]

### 3. Filtering a valued network: four options, four different networks


In [ ]:
## (a) absolute threshold - simple, arbitrary, biased against small countries
f_abs <- el22[weight >= 5000]

## (b) top-k destinations of each country - guarantees every node survives
f_topk <- el22[order(from, -weight)][, head(.SD, 5), by = from]

## (c) share of the source's total exports of value added
f_share <- el22[, .SD[weight / sum(weight) >= 0.05], by = from]

## (d) DISPARITY FILTER (Serrano, Boguna & Vespignani 2009, PNAS) - keeps the
##     links that are significantly stronger than a random allocation of a
##     node's strength across its ties.
disparity_filter <- function(el, alpha = 0.05) {
  d <- copy(as.data.table(el))
  d[, `:=`(k = .N, s = sum(weight)), by = from]
  d[, p_ij := weight / s]
  d[, alpha_ij := (1 - p_ij)^(k - 1)]              # p-value under the null
  d[k > 1 & alpha_ij < alpha]
}
f_disp <- disparity_filter(el22, alpha = 0.01)

rbindlist(list(
  data.table(filter = "none",            edges = nrow(el22)),
  data.table(filter = "weight >= 5bn",   edges = nrow(f_abs)),
  data.table(filter = "top 5 per country", edges = nrow(f_topk)),
  data.table(filter = ">= 5% of exports", edges = nrow(f_share)),
  data.table(filter = "disparity a=0.01", edges = nrow(f_disp))))

## Does the choice change the answer? Compare betweenness rankings.
bt <- function(el) {
  gg <- graph_from_data_frame(el[, .(from, to, weight)], directed = TRUE,
                              vertices = reg[, .(name = iso3)])
  betweenness(gg, weights = NA)
}
btw <- data.table(country = reg$iso3, abs5 = bt(f_abs), topk = bt(f_topk),
                  share5 = bt(f_share), disparity = bt(f_disp))
round(cor(btw[, -1], method = "spearman"), 2)
btw[order(-disparity)][1:10]

## The absolute threshold makes small open economies disappear; the disparity
## filter keeps them. State the filter in the paper - it IS a modelling choice.

### 4. Map it (you can only draw a valued network after filtering it)


In [ ]:
gp <- graph_from_data_frame(f_disp[, .(from, to, weight)], directed = TRUE,
                            vertices = reg[, .(name = iso3, region)])
gp <- induced_subgraph(gp, V(gp)[degree(gp) > 0])
V(gp)$out_str <- strength(gp, mode = "out")

ggraph(gp, layout = "stress") +
  geom_edge_fan(aes(edge_width = weight, edge_alpha = weight),
                edge_colour = "grey55",
                arrow = arrow(angle = 15, length = unit(0.09, "inches"),
                              type = "closed"),
                start_cap = circle(2, "mm"), end_cap = circle(3, "mm")) +
  scale_edge_width(range = c(0.1, 1.6)) + scale_edge_alpha(range = c(0.15, 0.7)) +
  geom_node_point(aes(size = out_str, fill = region), shape = 21, colour = "white") +
  geom_node_text(aes(label = name), size = 2.6, repel = TRUE) +
  scale_size(range = c(2, 12)) +
  theme_graph(base_family = "sans") +
  labs(title = "Value added embodied in foreign final demand, 2022",
       subtitle = "OECD TiVA, disparity-filter backbone (alpha = 0.01)",
       fill = "region") + guides(size = "none", edge_width = "none",
                                 edge_alpha = "none")

### 5. Communities = trade blocs, and how regional they are


In [ ]:
## Louvain needs an undirected graph: symmetrise the flows (i<->j = i->j + j->i)
symmetrise <- function(el) {
  d <- copy(as.data.table(el))
  d[, `:=`(a = pmin(from, to), b = pmax(from, to))]
  d[, .(weight = sum(weight)), by = .(from = a, to = b)]
}

sym22 <- symmetrise(el22)
gu <- graph_from_data_frame(sym22, directed = FALSE,
                           vertices = reg[, .(name = iso3, region)])
cl_raw <- cluster_louvain(gu, weights = E(gu)$weight)
c(blocs = length(unique(membership(cl_raw))),
  modularity = round(modularity(cl_raw), 3),
  largest_share = round(max(table(membership(cl_raw))) / vcount(gu), 2))

## Two blocks, and they are essentially "around the USA" and "around Germany".
## On a COMPLETE VALUED network, modularity is driven by the size of the nodes:
## big economies trade a lot with everybody, so the partition mostly recovers
## who is big. Before looking for structure, take size out.

## Revealed trade intensity: observed flow / flow expected from the two
## countries' sizes (the same normalisation logic as the CORDIS index in block 2)
normalise <- function(sym) {
  d <- copy(sym)
  str <- rbind(d[, .(c = from, w = weight)], d[, .(c = to, w = weight)])[
    , .(s = sum(w)), by = c]
  W <- sum(d$weight)
  d <- merge(merge(d, str, by.x = "from", by.y = "c"),
             str, by.x = "to", by.y = "c", suffixes = c("_f", "_t"))
  d[, weight := weight / (s_f * s_t / (2 * W))]
  d[, .(from, to, weight)]
}

blocs <- function(yr, normalised = TRUE) {
  e <- va[year == yr, .(from = source, to = destination, weight = va_musd)]
  sym <- symmetrise(e)
  if (normalised) sym <- normalise(sym)
  gg <- graph_from_data_frame(sym, directed = FALSE,
                              vertices = reg[, .(name = iso3, region)])
  cl <- cluster_louvain(gg, weights = E(gg)$weight)
  list(g = gg, cl = cl,
       stats = data.table(year = yr, normalised = normalised,
                          blocs = length(unique(membership(cl))),
                          modularity = round(modularity(cl), 3),
                          nmi_with_geography = round(
                            compare(membership(cl), as.integer(factor(V(gg)$region)),
                                    method = "nmi"), 3)))
}

rbind(blocs(1995, normalised = FALSE)$stats, blocs(2022, normalised = FALSE)$stats,
      blocs(1995)$stats, blocs(2022)$stats)

## With raw weights: two blocks, no change in 27 years - the size effect swamps
## everything. With normalised weights: more, smaller blocks that align much more
## closely with geography, and now the 1995 -> 2022 comparison is informative.
## This is exactly the question in Fusillo, Montresor & Vittucci Marzetti (2024):
## have national and regional boundaries really faded away?

b22 <- blocs(2022)
memb <- data.table(iso3 = V(b22$g)$name, region = V(b22$g)$region,
                   bloc = as.integer(membership(b22$cl)))
va_out <- va[year == 2022, .(va = sum(va_musd)), by = .(iso3 = source)]
memb <- merge(memb, va_out, by = "iso3")
memb[, .(members = .N, va_bn = round(sum(va) / 1000),
         regions = uniqueN(region),
         core = paste(head(iso3[order(-va)], 4), collapse = " ")),
     by = bloc][order(-va_bn)]

## Communities on the DIRECTED graph instead (infomap follows the flow):
im <- cluster_infomap(g22, e.weights = E(g22)$weight)
c(louvain_blocs = length(unique(membership(b22$cl))),
  infomap_blocs = length(unique(membership(im))),
  agreement_nmi = round(compare(membership(b22$cl), membership(im), method = "nmi"), 3))

## Infomap puts everything in one module: a random walk on a complete weighted
## digraph never gets trapped anywhere. Not a bug - a property of the data.
## See 06_brokerage_communities.R for how to choose an algorithm and report it.

### 6. Brokerage in value chains: who intermediates between regions?


In [ ]:
## Here the data are DIRECTED, so "gatekeeper" (controls what enters my region)
## and "representative" (controls what leaves it) are finally different things.
## Run it on the disparity backbone: raw counts on a complete graph are meaningless.
## a slightly looser backbone (alpha = 0.05) leaves enough 2-paths to classify
gb <- graph_from_data_frame(disparity_filter(el22, alpha = 0.05)[, .(from, to, weight)],
                            directed = TRUE, vertices = reg[, .(name = iso3, region)])
gb <- induced_subgraph(gb, V(gb)[degree(gb) > 0])
c(nodes = vcount(gb), edges = ecount(gb))

roles <- brokerage_roles(gb, V(gb)$region)
roles <- merge(roles, data.table(name = V(gb)$name, region = V(gb)$region,
                                 out_str = strength(gb, mode = "out")), by = "name")
head(roles[order(-liaison)], 10)[, .(name, region, coordinator, gatekeeper,
                                     representative, consultant, liaison)]

## normalise by degree: who brokers MORE than their size implies?
roles[, total := coordinator + gatekeeper + representative + consultant + liaison]
head(roles[total >= 10][order(-liaison / total)], 10)[,
     .(name, region, share_liaison = round(liaison / total, 2),
       share_gatekeeper = round(gatekeeper / total, 2),
       share_coordinator = round(coordinator / total, 2), total)]
## Small open economies and re-export hubs (NLD, BEL, SGP, HKG, MEX) live off
## intermediation; the large ones broker within their own bloc.

### 7. IF WE HAVE TIME - one method, eight industries


In [ ]:
## The same pipeline, run per industry: the geography of value chains is not the
## same for food, cars, electronics and business services.
ind <- fread(daisy_data("tiva_va_by_industry.csv.gz"))
lab <- fread(daisy_data("tiva_industry_labels.csv"))
ind <- ind[source != destination]

by_ind <- rbindlist(lapply(unique(ind$industry), function(i) {
  e  <- ind[industry == i, .(from = source, to = destination, weight = va_musd)]
  gu <- graph_from_data_frame(normalise(symmetrise(e)), directed = FALSE,
                             vertices = reg[, .(name = iso3, region)])
  cl <- cluster_louvain(gu, weights = E(gu)$weight)
  s  <- e[, .(w = sum(weight)), by = from][match(V(gu)$name, from), w]
  data.table(industry = i,
             va_bn = round(sum(e$weight) / 1000),
             top3 = paste(V(gu)$name[order(-s)][1:3], collapse = " "),
             hhi = round(sum((s / sum(s))^2), 3),
             blocs = length(unique(membership(cl))),
             modularity = round(modularity(cl), 3),
             nmi_geography = round(compare(membership(cl),
                                   as.integer(factor(V(gu)$region)), "nmi"), 3))
}))
merge(by_ind, lab, by = "industry")[order(-va_bn)]
## Compare "modularity" (how bloc-structured the industry is) with
## "nmi_geography" (whether those blocs are geographic). Electronics and
## transport equipment are the regionalised ones; services are not.

### 8. IF WE HAVE TIME - gross trade or value added? (BACI vs TiVA)


In [ ]:
## BACI (CEPII) records gross bilateral flows of goods by HS6 product: what
## crosses the border. TiVA records where the value was actually created. For
## some countries the two tell very different stories - and the difference IS
## the network position.
bac <- fread(daisy_data("baci_bilateral_2023.csv.gz"))
bac[order(-exports_musd)][1:6]

gross <- bac[exporter != importer, .(gross_bn = sum(exports_musd) / 1000), by = .(iso3 = exporter)]
vadd  <- va[year == 2022, .(va_bn = sum(va_musd) / 1000), by = .(iso3 = source)]
cmp   <- merge(gross, vadd, by = "iso3")
cmp[, ratio := round(va_bn / gross_bn, 2)]          # VA generated per $ exported
cor(cmp$gross_bn, cmp$va_bn, method = "spearman")

cmp[gross_bn > 100][order(ratio)][1:10]             # pure transit / assembly
cmp[gross_bn > 100][order(-ratio)][1:10]            # value created at home
## Low ratio = you ship a lot but much of the value is foreign (Vietnam, Mexico,
## Belgium, Netherlands: assembly platforms and re-export hubs). Interpreting a
## gross-trade network as a network of "who produces what" is a measurement error
## with a name: double counting.

## The same contrast at the level of a single tie
gr_pairs <- bac[, .(a = pmin(exporter, importer), b = pmax(exporter, importer),
                    gross = exports_musd)][, .(gross = sum(gross)), by = .(a, b)]
va_pairs <- dy[, .(a, b, va = a_to_b + b_to_a)]
pairs <- merge(gr_pairs, va_pairs, by = c("a", "b"))
pairs[va > 20000][, ratio := round(va / gross, 2)][order(ratio)][1:8]

### 9. And the product space


In [ ]:
## The exporter x product matrix of BACI feeds exactly the machinery of
## 04_indicators.R: revealed comparative advantage, proximity between products,
## the product space, and the complexity indices. We build it there, so that the
## three category systems of this session - technologies (patents), topics
## (CORDIS) and products (trade) - sit side by side in one script.

---

## BLOCK 5 of the session (~20 min) - NETWORKS AS MEASUREMENT DEVICES

("indirect" uses: when you do not study the network, you use it to build a
variable)
Idea: a knowledge base has a *co-relational* structure. Represent technologies
as nodes and their joint use as links, and the network becomes a measurement
instrument for concepts that have no direct observable counterpart:
relatedness, variety, coherence, complexity, diversification potential.
Data: OECD REGPAT green patents (Y02/Y04S), EU NUTS regions, CPC 4-digit
classes, two periods (2010-2014, 2015-2019).


### 1. The regional technology portfolios


In [ ]:
rt  <- fread(daisy_data("green_region_tech_EU.csv.gz"))
def <- fread(daisy_data("cpc4_def.csv"))              # CPC4 labels
rt

## fractional counts: a patent is split across its regions and its CPC classes,
## so that every patent contributes exactly 1 to the world total
rt[, sum(n_pat), by = period]

## NUTS-3 -> NUTS-2 (the level at which regional innovation is usually studied)
rt[, nuts2 := substr(reg_code, 1, 4)]
reg_tech <- rt[, .(n_pat = sum(n_pat)), by = .(nuts2, ctry_code, cpc4, period)]

## keep the recent period and regions/technologies with enough mass
d <- reg_tech[period == "2015-2019"]
big_reg  <- d[, .(tot = sum(n_pat)), by = nuts2][tot >= 20, nuts2]
big_tech <- d[, .(tot = sum(n_pat)), by = cpc4][tot >= 20, cpc4]
d <- d[nuts2 %in% big_reg & cpc4 %in% big_tech]
d[, .(regions = uniqueN(nuts2), technologies = uniqueN(cpc4))]

## region x technology matrix
X <- as.matrix(dcast(d, nuts2 ~ cpc4, value.var = "n_pat", fill = 0),
               rownames = "nuts2")
dim(X)

### 2. Revealed Technological Advantage: from counts to specialisation


In [ ]:
## RTA_rt = (X_rt / X_r.) / (X_.t / X_..)   ("Balassa index")
RTA <- (X / rowSums(X)) / rep(colSums(X) / sum(X), each = nrow(X))
M   <- (RTA >= 1) * 1                        # binary specialisation matrix
mean(M)                                      # density of the two-mode network

diversity <- rowSums(M)                      # n. of technologies of a region
ubiquity  <- colSums(M)                      # n. of regions with that technology
sort(diversity, decreasing = TRUE)[1:10]
sort(ubiquity, decreasing = TRUE)[1:10]

merge(data.table(cpc4 = names(ubiquity), ubiquity), def,
      by = "cpc4")[order(ubiquity)][1:8]                # most exclusive classes

### 3. Two ways of measuring RELATEDNESS between technologies


In [ ]:
## (A) co-classification inside patents (the "knowledge space" proper):
##     two classes are related if inventors combine them in the same document
cooc <- fread(daisy_data("green_tech_cooc_EU.csv.gz"))
npat <- fread(daisy_data("green_tech_npat_EU.csv"))
techs <- colnames(X)
cooc  <- cooc[cpc4_i %in% techs & cpc4_j %in% techs]
C <- sparseMatrix(i = match(cooc$cpc4_i, techs), j = match(cooc$cpc4_j, techs),
                  x = cooc$n_cooc, dims = c(length(techs), length(techs)),
                  dimnames = list(techs, techs), symmetric = FALSE)
C <- as.matrix(C + t(C))
n_t <- setNames(npat$n_pat, npat$cpc4)[techs]
## association strength (Van Eck & Waltman): observed / expected co-occurrence
Phi_pat <- C / outer(n_t, n_t) * sum(n_t) / 2
Phi_pat[!is.finite(Phi_pat)] <- 0

## (B) co-specialisation across regions (Hidalgo et al. "proximity"):
##     two technologies are related if the same regions are good at both
Co <- t(M) %*% M
Phi_reg <- Co / outer(ubiquity, ubiquity, pmax)      # min conditional probability
Phi_reg[!is.finite(Phi_reg)] <- 0
diag(Phi_reg) <- 0

## Do the two measures agree? (they answer different questions!)
iu <- upper.tri(Phi_reg)
cor(Phi_pat[iu], Phi_reg[iu], method = "spearman")

### 4. The green knowledge space, drawn


In [ ]:
## keep the strongest links only, otherwise the map is a hairball
thr <- quantile(Phi_pat[iu], 0.98)
A <- Phi_pat * (Phi_pat >= thr)
g_ks <- graph_from_adjacency_matrix(A, mode = "undirected", weighted = TRUE, diag = FALSE)
g_ks <- induced_subgraph(g_ks, V(g_ks)[degree(g_ks) > 0])
V(g_ks)$patents <- n_t[V(g_ks)$name]
V(g_ks)$label   <- def$label[match(V(g_ks)$name, def$cpc4)]
V(g_ks)$comm    <- membership(cluster_louvain(g_ks, weights = E(g_ks)$weight))
g_ks

## which technologies bridge the green knowledge space? (candidate GPTs)
sort(betweenness(g_ks, weights = NA), decreasing = TRUE)[1:10]

ggraph(g_ks, layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey85") +
  scale_edge_width(range = c(0.1, 1.5)) +
  geom_node_point(aes(size = patents, fill = factor(comm)), shape = 21, colour = "white") +
  geom_node_text(aes(label = name), size = 2.4, repel = TRUE, max.overlaps = 30) +
  scale_size(range = c(1, 10)) +
  theme_graph(base_family = "sans") + theme(legend.position = "none") +
  labs(title = "Green knowledge space (CPC4 co-classification, EU 2015-2019)")

### 5. Region-level indicators built ON the network


In [ ]:
sh <- X / rowSums(X)                  # patent shares of each region

## (a) VARIETY = entropy of the portfolio, decomposed into related (within
##     3-digit CPC groups) and unrelated (between groups) variety
grp <- substr(colnames(X), 1, 3)
H <- function(p) { p <- p[p > 0]; -sum(p * log2(p)) }
variety <- apply(sh, 1, H)
grp_sh  <- t(rowsum(t(sh), grp))                      # shares by 3-digit group
unrelated <- apply(grp_sh, 1, H)                      # between-group entropy
related   <- variety - unrelated                      # within-group entropy

## (b) COHERENCE = average relatedness of the technologies a region holds,
##     weighted by their importance in the portfolio (Nesta & Saviotti)
coherence <- as.numeric(rowSums((sh %*% Phi_pat) * sh))

## (c) RELATEDNESS DENSITY of the technologies the region is NOT (yet) in:
##     how "close" is a new technology to what the region already does?
dens <- (M %*% Phi_reg) / rep(colSums(Phi_reg), each = nrow(M)) * 100
avg_density_out <- rowSums(dens * (1 - M)) / rowSums(1 - M)
## CAVEAT: averaged over a region, relatedness density is almost collinear with
## diversity (see the correlation matrix below). The informative variation is at
## the region x technology level


## (d) COMPLEXITY of the regional portfolio
##     Hidalgo & Hausmann (2009) - applied to technologies by Balland & Rigby
##     (2017). Two equivalent readings of the same bipartite network:
##
##  d.1 METHOD OF REFLECTIONS - iterate "average of my neighbours' average"
reflections <- function(M, iter = 2) {
  kr <- rowSums(M); kt <- colSums(M)
  for (i in seq_len(iter)) {
    kr_new <- as.numeric((M %*% kt) / rowSums(M))
    kt_new <- as.numeric((t(M) %*% kr) / colSums(M))
    kr <- kr_new; kt <- kt_new
  }
  list(kr = setNames(kr, rownames(M)), kt = setNames(kt, colnames(M)))
}
## careful with the parity of the iteration: odd orders measure average
## UBIQUITY (high = simple), even orders average DIVERSITY (high = complex)
r1 <- reflections(M, 1); r2 <- reflections(M, 2)
c(order1_vs_diversity = cor(r1$kr, diversity),
  order2_vs_diversity = cor(r2$kr, diversity))

##  d.2 EIGENVECTOR FORM (what the Atlas of Economic Complexity computes)
##      Mtilde = D^-1 M U^-1 M' ; complexity = 2nd eigenvector, standardised.
##      complexity() in 00_setup.R does it for both sides of the matrix at once -
##      open it and read the SIGN conventions, they are where mistakes happen.
cx  <- complexity(M)
kci <- cx$actor        # regional knowledge complexity
tci <- cx$category     # technological complexity
c(kci_vs_reflections = round(cor(kci[names(r2$kr)], r2$kr), 2),
  tci_vs_ubiquity    = round(cor(tci, ubiquity[names(tci)]), 2))
## the second correlation must be NEGATIVE: complex technologies are held by few
## regions. If it is positive, your sign convention is upside down.

## most and least complex green technologies - and their ubiquity, to show that
## complexity is NOT just the inverse of ubiquity: it is second-order. A class
## held by few regions that are themselves poorly diversified is not complex.
tech_cx <- merge(data.table(cpc4 = names(tci), tci,
                            ubiquity = ubiquity[names(tci)]), def, by = "cpc4")
tech_cx[order(-tci)][1:6, .(cpc4, tci = round(tci, 2), ubiquity, label)]
tech_cx[order(tci)][1:6,  .(cpc4, tci = round(tci, 2), ubiquity, label)]

## put everything together: one row per region, ready for a regression
ind <- data.table(nuts2 = rownames(X),
                  patents = rowSums(X),
                  diversity, variety, related_variety = related,
                  unrelated_variety = unrelated,
                  coherence, relatedness_density = avg_density_out,
                  complexity = kci[rownames(X)])
ind[, country := substr(nuts2, 1, 2)]
ind[order(-complexity)][1:12]
round(cor(ind[, .(patents, diversity, variety, related_variety,
                  unrelated_variety, coherence, relatedness_density, complexity)]), 2)

fwrite(ind, "output_region_knowledge_indicators.csv")

ggplot(ind, aes(log(patents), complexity, label = nuts2)) +
  geom_point(aes(size = variety), alpha = .6, colour = "#2c7fb8") +
  geom_text(size = 2.4, vjust = -1, check_overlap = TRUE) +
  labs(x = "log green patents", y = "complexity of the portfolio (KCI)",
       size = "variety") + theme_minimal()

### 6. IF WE HAVE TIME - does relatedness predict diversification?


In [ ]:
## The canonical evolutionary-economic-geography test: regions enter new
## technologies that are related to what they already do. We have two periods.
d0 <- reg_tech[period == "2010-2014" & nuts2 %in% rownames(X) & cpc4 %in% colnames(X)]
X0 <- matrix(0, nrow(X), ncol(X), dimnames = dimnames(X))
X0[cbind(d0$nuts2, d0$cpc4)] <- d0$n_pat
RTA0 <- (X0 / pmax(rowSums(X0), 1)) / rep(colSums(X0) / sum(X0), each = nrow(X0))
M0 <- (RTA0 >= 1) * 1
M0[!is.finite(M0)] <- 0

Co0 <- t(M0) %*% M0
Phi0 <- Co0 / outer(pmax(colSums(M0), 1), pmax(colSums(M0), 1), pmax)
Phi0[!is.finite(Phi0)] <- 0; diag(Phi0) <- 0
dens0 <- (M0 %*% Phi0) / rep(pmax(colSums(Phi0), 1e-9), each = nrow(M0)) * 100

entry <- data.table(
  nuts2 = rep(rownames(M), times = ncol(M)),
  cpc4  = rep(colnames(M), each = nrow(M)),
  had   = as.vector(M0), has = as.vector(M),
  density0 = as.vector(dens0))
entry <- entry[had == 0]                       # only technologies NOT held before
entry[, entered := as.integer(has == 1)]
entry[, mean(entered), by = .(density_bin = cut(density0, breaks = c(-1, 5, 10, 20, 100)))][order(density_bin)]

summary(glm(entered ~ density0, data = entry, family = binomial))$coefficients
## => the probability of entering a new green technology increases with the
##    relatedness density of that technology to the regional portfolio: the
##    network is the measurement device behind the "principle of relatedness".

### 7. IF WE HAVE TIME - the same construction, other category systems


In [ ]:
## Nothing above was specific to patents. The knowledge space needed only
## (i) documents and (ii) categories attached to them; the indicators needed only
## an actor x category matrix. Change the category system and the same code maps
## a different domain. Two examples, from the data of blocks 2 and 4.

## --- 7a. TOPICS: the thematic space of EU climate research ---------------- ##
## euroSciVoc classifies every Horizon Europe project into scientific fields;
## co-occurrence of fields within a project plays the role of co-classification
## of CPC codes within a patent.
sv <- fread(daisy_data("cordis_he_scivoc.csv.gz"))
sv[, .(projects = uniqueN(project_id), fields = uniqueN(sci_voc),
       fields_per_project = round(.N / uniqueN(project_id), 1))]

g_topic <- make_net(proj_two_mode(sv, "project_id", "sci_voc"))
g_topic <- delete_edges(g_topic, E(g_topic)[weight < 10])
g_topic <- induced_subgraph(g_topic, V(g_topic)[degree(g_topic) > 0])
V(g_topic)$comm <- membership(cluster_louvain(g_topic, weights = E(g_topic)$weight))
sort(degree(g_topic), decreasing = TRUE)[1:12]

## which fields bridge otherwise separate research areas?
sort(betweenness(g_topic, weights = NA), decreasing = TRUE)[1:8]

ggraph(g_topic, layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey85") +
  scale_edge_width(range = c(0.1, 2)) +
  geom_node_point(aes(size = n_events, fill = factor(comm)), shape = 21,
                  colour = "white") +
  geom_node_text(aes(label = name), size = 2.6, repel = TRUE, max.overlaps = 20) +
  scale_size(range = c(1, 9)) +
  theme_graph(base_family = "sans") + theme(legend.position = "none") +
  labs(title = "Thematic space of Horizon Europe climate projects (euroSciVoc)")

## --- 7b. PRODUCTS: the product space and economic complexity -------------- ##
## The original application (Hidalgo et al. 2007): actors are countries,
## categories are exported products. Same four steps as sections 2-5.
## NOTE the colClasses: HS codes have leading zeros ("0101" is horses), and
## fread would happily turn them into the integer 101. Classification codes are
## always character - this bug has ruined more than one paper.
cp <- fread(daisy_data("baci_country_product_2023.csv.gz"),
            colClasses = c(hs4 = "character"))
hs <- fread(daisy_data("hs4_labels.csv"),
            colClasses = c(hs4 = "character", hs2 = "character"))

## the "p" suffix keeps the patent objects of the previous sections available
Xp <- as.matrix(dcast(cp, exporter ~ hs4, value.var = "exports_musd", fill = 0),
                rownames = "exporter")
RCAp <- (Xp / rowSums(Xp)) / rep(colSums(Xp) / sum(Xp), each = nrow(Xp))
Mp   <- (RCAp >= 1) * 1
c(countries = nrow(Mp), products = ncol(Mp), density = round(mean(Mp), 3))
sort(rowSums(Mp), decreasing = TRUE)[1:8]           # most diversified exporters

## proximity a la Hidalgo et al. (2007): min conditional probability
Cop  <- t(Mp) %*% Mp
Phip <- Cop / outer(colSums(Mp), colSums(Mp), pmax)
Phip[!is.finite(Phip)] <- 0; diag(Phip) <- 0

## complexity: the same helper as in section 5
cxp <- complexity(Mp)
eci <- cxp$actor; pci <- cxp$category
c(eci_vs_diversity = round(cor(eci, cxp$diversity), 2),
  pci_vs_ubiquity  = round(cor(pci, cxp$ubiquity), 2))     # must be negative
head(sort(eci, decreasing = TRUE), 10)                     # most complex economies
merge(data.table(hs4 = names(pci), pci), hs, by = "hs4")[order(-pci)][1:6,
      .(hs4, label, pci = round(pci, 2))]
merge(data.table(hs4 = names(pci), pci), hs, by = "hs4")[order(pci)][1:6,
      .(hs4, label, pci = round(pci, 2))]

## the product space: strongest links only, colour = product complexity
thr_p <- quantile(Phip[upper.tri(Phip)], 0.995)
g_prod <- graph_from_adjacency_matrix(Phip * (Phip >= thr_p), mode = "undirected",
                                      weighted = TRUE, diag = FALSE)
g_prod <- induced_subgraph(g_prod, V(g_prod)[degree(g_prod) > 0])
V(g_prod)$exports <- colSums(Xp)[V(g_prod)$name]
V(g_prod)$pci     <- pci[V(g_prod)$name]
V(g_prod)$hs2     <- substr(V(g_prod)$name, 1, 2)
c(nodes = vcount(g_prod), edges = ecount(g_prod))

ggraph(g_prod, layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey82") +
  scale_edge_width(range = c(0.1, 1.2)) +
  geom_node_point(aes(size = exports, fill = pci), shape = 21, colour = "white") +
  scale_fill_gradient2(low = "#2c7fb8", mid = "grey90", high = "#d95f0e",
                       midpoint = 0) +
  geom_node_text(aes(label = ifelse(rank(-exports) <= 30, name, "")), size = 2.6,
                 repel = TRUE, max.overlaps = 25) +
  scale_size(range = c(1, 11)) +
  theme_graph(base_family = "sans") + guides(size = "none") +
  labs(title = "The product space, BACI 2023 (HS4)",
       subtitle = "strongest 0.5% of proximity links; colour = product complexity",
       fill = "PCI")

## --- 7c. Three spaces, one construction ---------------------------------- ##
space_summary <- function(g, what) data.table(
  space = what, nodes = vcount(g), edges = ecount(g),
  density = round(edge_density(g), 3),
  communities = length(unique(membership(cluster_louvain(g, weights = E(g)$weight)))),
  most_central = V(g)$name[which.max(degree(g))])
rbind(space_summary(g_ks,    "technologies (CPC4, patents)"),
      space_summary(g_topic, "topics (euroSciVoc, projects)"),
      space_summary(g_prod,  "products (HS4, exports)"))

## Same four lines of code, three literatures: the knowledge space (Krafft,
## Quatraro & Saviotti), the map of research fields (bibliometrics), the product
## space (Hidalgo, Hausmann). What changes is the category system you believe in.

---

## BLOCK 6 - BROKERAGE AND COMMUNITY DETECTION: a toolbox with hints

(reference material: the ideas are used live in blocks 2 and 4, the menu of
algorithms and benchmarks is here for when you write your own paper)
Two questions come back in every seminar: "who is the broker?" and "how do I
find groups?". Both have several answers, and the answers do not agree. This
script is the reference: what each measure actually counts, how to compute it
when igraph has no function for it, and how to report it so that a referee
cannot ask "what would happen with another algorithm?".
It runs on the networks built in blocks 2 and 5, but every function here works
on any igraph object.


### PART A - BROKERAGE


In [ ]:
## Four different ideas travel under the same word:
##
##   1. BETWEENNESS         - how often you sit on shortest paths (global flow)
##   2. STRUCTURAL HOLES    - Burt: are your contacts disconnected from each
##                            other? (constraint, effective size, efficiency)
##   3. GOULD-FERNANDEZ     - what KIND of gap do you bridge, given a group
##      brokerage roles       partition (coordinator, gatekeeper, representative,
##                            consultant, liaison)
##   4. E-I INDEX           - how outward-looking are your ties (group level)
##
## They answer different questions. Pick the one that matches your theory, and
## say why. A paper that reports "betweenness" when the argument is about
## bridging two communities is answering the wrong question.

part <- fread(daisy_data("cordis_he_participants.csv.gz"))
g_org <- make_net(proj_two_mode(part, "project_id", "org_id", max_size = 40),
                  node_attr = part[, .(org_name = org_name[1], country = country[1],
                                       type = activity_type[1]), by = org_id],
                  by = "org_id")

## Work on a manageable subgraph. Brokerage needs to look at every 2-path, so
## cost grows with degree^2: on the full 15,000-organisation network the roles
## below take minutes, on the 2,400 most connected ones a few seconds.
g <- induced_subgraph(g_org, V(g_org)[degree(g_org) >= 50])
g <- giant(g)
c(nodes = vcount(g), edges = ecount(g), mean_degree = round(mean(degree(g))))

## --- A1. Burt's ego-network measures --------------------------------------- ##
## igraph gives you constraint(); effective_size() is one of the helpers defined
## in 00_setup.R - open it, it is five lines of sparse-matrix algebra.
V(g)$constraint <- constraint(g)                     # LOW  = many holes
V(g)$eff_size   <- effective_size(g)                 # HIGH = many non-redundant
V(g)$efficiency <- V(g)$eff_size / degree(g)         # per-contact yield
V(g)$betw       <- betweenness(g, weights = NA, normalized = TRUE)

brok <- as.data.table(as_data_frame(g, what = "vertices"))
brok[, degree := degree(g)]
round(cor(brok[, .(degree, betw, constraint, eff_size, efficiency)]), 2)
## note: effective size correlates ~1 with degree, efficiency does not. If you
## want "brokerage net of size", use efficiency or constraint, not effective size.

brok[order(constraint)][1:10, .(org_name, country, type, degree,
                                constraint = round(constraint, 3),
                                efficiency = round(efficiency, 2))]

## --- A2. Gould & Fernandez brokerage roles --------------------------------- ##
## A node v brokers a 2-path i -> v -> j when i and j are NOT directly tied.
## Given a partition into groups, the role depends on where i, v and j sit:
##
##   coordinator     i, v, j all in the same group        (within-group broker)
##   gatekeeper      i outside, v and j inside            (controls entry)
##   representative  i and v inside, j outside            (controls exit)
##   consultant      i and j in the SAME other group      (itinerant broker)
##   liaison         i, v, j all in different groups      (bridges two others)
##
## Not in igraph. sna::brokerage() does it (with intergraph::asNetwork), but
## brokerage_roles() in 00_setup.R is transparent, has no dependency and runs on
## 10^3-10^4 nodes. Read the function before using it: the two corrections it
## applies (dropping i == j, and dropping pairs that are already tied) are the
## whole definition.

## NOTE on undirected networks: with i -> v -> j read in both directions,
## "gatekeeper" and "representative" are the same count by construction (see the
## table below). Only directed data - citations, trade flows, supply chains -
## separates controlling entry from controlling exit.

## groups = country: who mediates between national research systems?
roles <- brokerage_roles(g, V(g)$country)
roles <- merge(roles, brok[, .(name, org_name, country, type, degree)], by = "name")
roles[order(-liaison)][1:10, .(org_name, country, type, degree,
                               coordinator, gatekeeper, representative,
                               consultant, liaison)]

## --- A3. Raw counts are useless without a benchmark ------------------------ ##
## Every count grows with degree. Compare each node with what it would score if
## the group labels were reshuffled at random (keeping the network fixed).
brokerage_z <- function(g, group, reps = 10) {
  obs <- brokerage_roles(g, group)
  cols <- setdiff(names(obs), "name")
  sims <- lapply(seq_len(reps), function(r)
    as.matrix(brokerage_roles(g, sample(group))[, ..cols]))
  mu <- Reduce(`+`, sims) / reps
  sdv <- sqrt(Reduce(`+`, lapply(sims, function(x) (x - mu)^2)) / max(reps - 1, 1))
  z <- (as.matrix(obs[, ..cols]) - mu) / pmax(sdv, 1e-9)
  cbind(obs[, .(name)], as.data.table(round(z, 2)))
}
## (10 permutations to keep the class moving - about 30 seconds; use 500+ in a
##  paper, and run it once overnight rather than in a loop you watch)
zz <- brokerage_z(g, V(g)$country, reps = 10)
zz <- merge(zz, brok[, .(name, org_name, country, degree)], by = "name")
zz[order(-liaison)][1:10, .(org_name, country, degree, gatekeeper,
                            representative, liaison)]
## Now "liaison" means "more than expected by chance", which is what the theory
## is about. The identity of the top brokers usually changes: show both tables.

## --- A4. Choosing ---------------------------------------------------------- ##
## - fragmented network (co-invention)? betweenness is dominated by component
##   structure; prefer ego-level measures (constraint, efficiency).
## - projected affiliation network? every event is a clique, so constraint is
##   mechanically high; compare against events of the same size.
## - theory about categories (countries, sectors, public/private)? that is
##   exactly Gould-Fernandez; report the z-scores, not the raw counts.
## - weighted networks: betweenness treats weights as DISTANCES (a strong tie is
##   a long detour!). Pass weights = 1/w, or weights = NA to ignore them.

### PART B - COMMUNITY DETECTION


In [ ]:
## No algorithm "finds the true communities": each optimises a different
## objective. What matters is that your groups are (i) reproducible,
## (ii) interpretable, (iii) not an artefact of one arbitrary choice.

gc <- giant(g)
W  <- E(gc)$weight

## --- B1. The main families ------------------------------------------------- ##
algos <- list(
  louvain      = function(x) cluster_louvain(x, weights = E(x)$weight),
  leiden_cpm   = function(x) cluster_leiden(x, objective_function = "CPM",
                                            resolution = 0.05, weights = E(x)$weight),
  fast_greedy  = function(x) cluster_fast_greedy(x, weights = E(x)$weight),
  walktrap     = function(x) cluster_walktrap(x, weights = E(x)$weight),
  infomap      = function(x) cluster_infomap(x, e.weights = E(x)$weight),
  label_prop   = function(x) cluster_label_prop(x, weights = E(x)$weight)
)
comp <- rbindlist(lapply(names(algos), function(a) {
  t0 <- proc.time()[["elapsed"]]
  cl <- algos[[a]](gc)
  data.table(algorithm = a, communities = length(unique(membership(cl))),
             modularity = round(modularity(gc, membership(cl), weights = W), 3),
             largest_share = round(max(table(membership(cl))) / vcount(gc), 2),
             seconds = round(proc.time()[["elapsed"]] - t0, 1))
}))
comp[order(-modularity)]
## In this dense projected network infomap collapses almost everything into one
## module and label propagation into two: both are designed for sparser graphs.
## That is information, not failure - report it instead of hiding it.
## Read this table before choosing: infomap tends to many small communities,
## modularity-based methods to few large ones, label propagation is fast but
## unstable, edge-betweenness (not run here) is O(n*m) - forget it above ~1,000 nodes.

## --- B2. Weights and direction matter -------------------------------------- ##
c(weighted   = modularity(gc, membership(cluster_louvain(gc, weights = W)), weights = W),
  unweighted = modularity(gc, membership(cluster_louvain(gc, weights = NA))))
## Decide explicitly: is a tie of weight 8 eight times "closer" than a tie of 1?
## For counts of shared documents, usually yes; for correlations, usually no.

## --- B3. The resolution parameter is a research choice --------------------- ##
res_scan <- rbindlist(lapply(c(0.5, 1, 2, 4), function(r) {
  cl <- cluster_louvain(gc, weights = W, resolution = r)
  data.table(resolution = r, communities = length(unique(membership(cl))),
             modularity = round(modularity(gc, membership(cl), weights = W), 3),
             largest_share = round(max(table(membership(cl))) / vcount(gc), 2))
}))
res_scan
## Modularity has a RESOLUTION LIMIT: it cannot see communities smaller than
## ~sqrt(2m). Scanning the resolution and reporting the range is the honest move.

## --- B4. Stability: run it again ------------------------------------------- ##
## Louvain and Leiden are stochastic. Are your communities a property of the
## network or of the seed?
parts <- lapply(1:20, function(s) { set.seed(s); membership(cluster_louvain(gc, weights = W)) })
mods  <- sapply(parts, function(m) modularity(gc, m, weights = W))
c(min = round(min(mods), 4), max = round(max(mods), 4),
  n_comm_min = min(sapply(parts, function(m) length(unique(m)))),
  n_comm_max = max(sapply(parts, function(m) length(unique(m)))))

## agreement between runs: NMI = 1 means identical partitions
nmi <- outer(seq_along(parts), seq_along(parts), Vectorize(function(i, j)
  compare(parts[[i]], parts[[j]], method = "nmi")))
round(c(mean_nmi = mean(nmi[upper.tri(nmi)]), min_nmi = min(nmi[upper.tri(nmi)])), 3)
## compare() also gives "adjusted.rand" and "vi" (variation of information).

## --- B5. Consensus: keep what survives ------------------------------------- ##
## co-membership frequency across runs, then cluster the consensus matrix
co <- Reduce(`+`, lapply(parts, function(m) outer(m, m, "==") * 1)) / length(parts)
mean(co[upper.tri(co)] > 0 & co[upper.tri(co)] < 1)   # share of unstable pairs
g_cons <- graph_from_adjacency_matrix(co * (co >= 0.9), mode = "undirected",
                                      weighted = TRUE, diag = FALSE)
cons <- components(g_cons)
c(consensus_groups = cons$no, largest = max(cons$csize))
## Nodes that never travel together are the ones you can safely talk about.

## --- B6. Interpretation is the actual result ------------------------------- ##
## A community is only useful if you can say what it IS. Cross it with attributes.
cl <- cluster_louvain(gc, weights = W)
memb <- data.table(name = V(gc)$name, country = V(gc)$country,
                   type = V(gc)$type, comm = as.integer(membership(cl)))
top <- memb[, .N, by = comm][order(-N)][1:5]$comm
memb[comm %in% top, .(orgs = .N,
                      top_country = names(which.max(table(country))),
                      country_hhi = round(sum(prop.table(table(country))^2), 3),
                      pct_private = round(100 * mean(type == "PRC")),
                      pct_academic = round(100 * mean(type %in% c("HES", "REC")))),
     by = comm][order(-orgs)]

## --- B7. What to report ---------------------------------------------------- ##
## In the paper, one sentence must contain: algorithm + implementation and
## version + weights used + resolution + seed/number of runs + modularity +
## number and size distribution of communities + one robustness check
## (another algorithm, or the consensus above). Anything less is not replicable.
##
## And remember what modularity cannot do: it always finds a partition, even in
## a random graph. Benchmark against a degree-preserving rewiring:
rnd <- rewire(gc, with = keeping_degseq(niter = 10 * ecount(gc)))
c(observed = round(modularity(gc, membership(cluster_louvain(gc, weights = NA))), 3),
  rewired  = round(modularity(rnd, membership(cluster_louvain(rnd))), 3))

---

## Exercises and the five-question checklist


In [ ]:

## ---------------------------------------------------------------------------
## 1. HOW FRAGILE ARE THE RANKINGS? (patents)
## ---------------------------------------------------------------------------
## Rebuild the green co-invention network keeping only patents with at most 10
## inventors, and compare the top-10 inventors by betweenness with the full
## network. How many names survive?
## Hint: proj_two_mode(inv, "appln_id", "person_id", max_size = 10)

## ---------------------------------------------------------------------------
## 2. WHO BRIDGES GREEN AND NON-GREEN TECHNOLOGY? (patents)
## ---------------------------------------------------------------------------
## Using pat_all_inventors_ITgreen.csv.gz, classify each inventor as green-only,
## non-green-only or mixed, and test whether "mixed" inventors have higher
## betweenness in the overall network. This is the empirical core of several
## papers on green technology recombination.
## Hint: inv_all[, .(green_share = mean(green)), by = person_id] then merge on
## the vertex table and compare distributions.

## ---------------------------------------------------------------------------
## 3. A NATIONAL SUBNETWORK (CORDIS)
## ---------------------------------------------------------------------------
## Take the Horizon Europe organisation network, extract the subgraph of Italian
## organisations, and find (a) the most central ones, (b) the share of their ties
## that stay inside Italy. Repeat for another country and compare.
## Hint: induced_subgraph(g, V(g)[country == "IT"]) for (a); for (b) work on the
## full edge list and use the country attribute of both endpoints.

## ---------------------------------------------------------------------------
## 4. MONEY AND POSITION (CORDIS)
## ---------------------------------------------------------------------------
## Aggregate the organisation network at country level and check whether
## betweenness in the country network is correlated with the EC contribution
## received per participation. Who punches above its weight?

## ---------------------------------------------------------------------------
## 5. YOUR OWN LITERATURE (OpenAlex)
## ---------------------------------------------------------------------------
## Change the query in 03_publications.R to the topic of your PhD, rebuild the
## institution network and identify the 10 most central institutions. Then look
## at them by hand: do you recognise duplicates or aggregation problems?

## ---------------------------------------------------------------------------
## 6. PERSISTENCE OF REGIONAL KNOWLEDGE STRUCTURES (indicators)
## ---------------------------------------------------------------------------
## Recompute variety, coherence and complexity for the period 2010-2014 and
## correlate them with the 2015-2019 values. Which indicator is most persistent?
## Then regress the growth of green patents 2015-2019 on the 2010-2014
## indicators. (Careful: this is a descriptive exercise, not a causal claim.)

## ---------------------------------------------------------------------------
## 7. FILTERS CHANGE FINDINGS (trade)
## ---------------------------------------------------------------------------
## Take the 2022 TiVA network and compute the top-10 countries by betweenness
## under the four filters of 05_trade.R plus one of your own (for instance, keep
## a link if it is above 2% of EITHER country's total). How many countries are in
## all five top-10 lists? Write the sentence you would put in a paper to justify
## your choice.

## ---------------------------------------------------------------------------
## 8. IS THE WORLD STILL REGIONAL? (trade)
## ---------------------------------------------------------------------------
## Run the bloc detection of 05_trade.R on all four years (1995, 2005, 2015,
## 2022) with normalised weights, and plot NMI-with-geography over time. Then do
## it for two industries separately (say C29_30 and J). Does "globalisation"
## look the same in cars and in software?

## ---------------------------------------------------------------------------
## 9. BROKERS OR HUBS? (any network)
## ---------------------------------------------------------------------------
## On the CORDIS organisation network, rank organisations by betweenness and by
## Gould-Fernandez liaison z-score (06_brokerage_communities.R). Take the ten
## largest rank differences and look them up: what kind of organisation gains,
## what kind loses? Which ranking would you use to test "brokerage improves
## innovation performance", and why?

## ---------------------------------------------------------------------------
## 10. HOW MUCH DO YOUR COMMUNITIES DEPEND ON THE ALGORITHM? (any network)
## ---------------------------------------------------------------------------
## Pick any network from the session. Detect communities with Louvain, Leiden
## (two resolutions), walktrap and Infomap; compute the pairwise NMI matrix;
## report the pair of algorithms that disagree most and inspect where they split.
## Then write the one-sentence methods note that would satisfy a referee.

## ---------------------------------------------------------------------------
## 11. STAY IN TWO MODES (advanced)
## ---------------------------------------------------------------------------
## Everything we did projected the two-mode network into one mode. Try instead
## to work directly on the bipartite graph: build it with the incidence matrix
## returned by proj_two_mode() (element $incidence), set the "type" attribute,
## and compute bipartite degree and clustering.
## Hint:
##   B  <- pr$incidence
##   gb <- graph_from_biadjacency_matrix(B)
##   table(V(gb)$type)
## Which measures still make sense? Which ones do not?

## ---------------------------------------------------------------------------
## CHECKLIST - the five questions to ask before you trust a network result
## ---------------------------------------------------------------------------
## 1. NODES     - are the actor identifiers disambiguated? (inventor names,
##                organisation ids, author ids). Errors in nodes are not noise:
##                they systematically split hubs and destroy paths.
## 2. TIES      - what does the tie mean? co-participation is not interaction.
##                Are mega-events (200-partner projects, 68-inventor patents)
##                creating cliques that drive your topology?
## 3. BOUNDARY  - which actors/events are in the population, and why? Country,
##                technology, sector and time-window choices all move the result.
## 4. TIME      - is the network a snapshot, a cumulative window, or a moving
##                window? Centrality is not comparable across window lengths.
## 5. INFERENCE - are you describing or estimating? Network measures are
##                generated regressors: they are endogenous to the same process
##                you are explaining, and they are correlated across units by
##                construction (no independence).